# 03 Data integration
Input: `flows_transformed.csv`. Outputs: train/test tables and an explicit feature plan.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src import config as cfg
from sklearn.model_selection import train_test_split

for folder in (cfg.DATA_PROCESSED, cfg.REPORTS): folder.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(cfg.DATA_PROCESSED / "flows_transformed.csv")
df[cfg.TIME] = pd.to_datetime(df[cfg.TIME])
numeric_features = list(cfg.NUMERIC_COLS)
categorical_features = [cfg.PROTOCOL]
identifiers = [cfg.SRC_IP, cfg.DST_IP]
port_cols = [cfg.SRC_PORT, cfg.DST_PORT]
datetime_cols = [cfg.TIME]
feature_candidates = numeric_features + categorical_features
context = ["row_id", cfg.TIME] + identifiers + port_cols
excluded = {cfg.LABEL: "target (kept separately as y)"}
if not cfg.INCLUDE_IDENTIFIERS:
    excluded.update({c: "identifier: excluded from model features" for c in identifiers})
excluded.update({c: "raw context: used for enrichment" for c in ["row_id", cfg.TIME] + port_cols})
summary = pd.DataFrame({"column": df.columns})
summary["status"] = summary["column"].map(lambda c: "excluded" if c in excluded else "feature candidate")
summary["reason"] = summary["column"].map(excluded).fillna("")
summary.to_csv(cfg.REPORTS / "excluded_columns.csv", index=False)

if cfg.SPLIT_STRATEGY == "time":
    ordered = df.sort_values(cfg.TIME)
    cut = int(len(ordered) * (1 - cfg.TEST_SIZE))
    train, test = ordered.iloc[:cut].copy(), ordered.iloc[cut:].copy()
else:
    train, test = train_test_split(df, test_size=cfg.TEST_SIZE, stratify=df[cfg.LABEL], random_state=cfg.RANDOM_STATE)
share = pd.DataFrame({"train": train[cfg.LABEL].value_counts(normalize=True), "test": test[cfg.LABEL].value_counts(normalize=True)}).round(4)
twin_cols = feature_candidates + context[1:]
twins = train[twin_cols].merge(test[twin_cols].drop_duplicates(), on=twin_cols, how="inner").shape[0]
plan = {"numeric_features": numeric_features, "categorical_features": categorical_features, "identifiers": identifiers,
        "port_cols": port_cols, "datetime": cfg.TIME, "features": feature_candidates, "context": context,
        "excluded": excluded, "split": {"strategy": cfg.SPLIT_STRATEGY, "test_size": cfg.TEST_SIZE,
        "random_state": cfg.RANDOM_STATE, "train_rows": len(train), "test_rows": len(test), "train_test_feature_twins": int(twins)}}
train.to_csv(cfg.DATA_PROCESSED / "train.csv", index=False)
test.to_csv(cfg.DATA_PROCESSED / "test.csv", index=False)
(cfg.DATA_PROCESSED / "feature_plan.json").write_text(json.dumps(plan, indent=2), encoding="utf-8")
print("train", train.shape, "test", test.shape, "twins", twins)
display(share)
display(summary)